
# ECF 2021 — Análisis de comportamiento financiero

**Notebook:** `2026-07-20_ECF_2021_02_Comportamiento_Analysis.ipynb`

## Objetivo general

Descubrir hallazgos sobre el comportamiento financiero de la población española que puedan:

1. aportar contexto nacional al análisis interno de **Banco Atlas**;
2. revelar patrones de ahorro, uso de canales, tenencia de productos, planificación y fragilidad;
3. identificar segmentos con necesidades financieras diferenciadas;
4. apoyar propuestas de adaptación o creación de productos, servicios y comunicaciones.

## Enfoque de trabajo

Este notebook comienza por la **Fase 1: preguntas, trazabilidad y exploración analítica sin visualizaciones**.

Antes de diseñar gráficos se hará lo siguiente:

- validar el nuevo master dataset amplio;
- conservar la equivalencia entre referencia de variable y pregunta original de la ECF;
- clasificar variables por temas de comportamiento financiero;
- formular preguntas de análisis conectadas con Banco Atlas;
- generar tablas de diagnóstico;
- localizar relaciones y segmentos candidatos;
- priorizar hallazgos para la fase de storytelling y visualización.

> Las asociaciones encontradas en la ECF son descriptivas. No deben interpretarse automáticamente como relaciones causales.


## 0. Librerías y configuración

In [20]:

from pathlib import Path
from itertools import combinations
import re
import warnings

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.max_colwidth", 180)

warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Rutas del proyecto

In [21]:

# Estructura esperada:
#
# ProjecteData/
# ├── Data/
# │   ├── 2026-07-20_ECF_2021_01_raw.dta
# │   ├── 2026-07-20_ECF_2021_02_MasterDataset.csv
# │   └── 2026-07-20_ECF_2021_02_MasterDataset_Diccionario.csv
# └── Scripts/
#     └── 2026-07-20_ECF_2021_02_Comportamiento_Analysis.ipynb

SCRIPTS_DIR = Path.cwd().resolve()
DATA_DIR = (SCRIPTS_DIR / "../Data").resolve()

MASTER_PATH = DATA_DIR / "2026-07-20_ECF_2021_02_MasterDataset.csv"
RAW_PATH = DATA_DIR / "2026-07-20_ECF_2021_01_raw.dta"
INDICATOR_DICTIONARY_PATH = (
    DATA_DIR / "2026-07-20_ECF_2021_02_MasterDataset_Diccionario.csv"
)

required_paths = {
    "Master dataset": MASTER_PATH,
    "Fichero raw Stata": RAW_PATH,
}

missing_paths = {
    name: path
    for name, path in required_paths.items()
    if not path.exists()
}

if missing_paths:
    print("Archivos no encontrados:")
    for name, path in missing_paths.items():
        print(f"  - {name}: {path}")
    raise FileNotFoundError(
        "Ejecuta el notebook desde ../Scripts/ y comprueba que "
        "los datos estén disponibles en ../Data/."
    )

print(f"Scripts: {SCRIPTS_DIR}")
print(f"Data:    {DATA_DIR}")
print(f"Master:  {MASTER_PATH.name}")


Scripts: /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Scripts
Data:    /Users/rogerdefez/Documents/Cursos i Llibres/BootCamp IT Academy/04_Simulador/ProjecteData/Equip_32/Data
Master:  2026-07-20_ECF_2021_02_MasterDataset.csv


## 2. Carga del master y metadatos originales de la ECF

In [22]:

df = pd.read_csv(MASTER_PATH, low_memory=False)

stata_reader = pd.io.stata.StataReader(RAW_PATH)
raw_variable_labels = stata_reader.variable_labels()
raw_value_label_names = list(stata_reader._lbllist)
raw_value_label_sets = stata_reader.value_labels()
raw_columns = list(raw_variable_labels)

indicator_dictionary = pd.DataFrame(
    columns=["Variable", "Definición"]
)

if INDICATOR_DICTIONARY_PATH.exists():
    indicator_dictionary = pd.read_csv(
        INDICATOR_DICTIONARY_PATH
    )
    indicator_dictionary.columns = [
        str(column).strip()
        for column in indicator_dictionary.columns
    ]

print(f"Observaciones: {df.shape[0]:,}")
print(f"Variables:     {df.shape[1]:,}")
print(f"Duplicados de fila: {df.duplicated().sum():,}")
print(
    "Columnas duplicadas:",
    int(df.columns.duplicated().sum()),
)

if df.shape[1] < 100:
    print(
        "\n⚠️ AVISO: el master contiene menos de 100 variables. "
        "Comprueba que has ejecutado la versión amplia del notebook "
        "de creación del master y que ha sustituido el CSV anterior."
    )


Observaciones: 7,764
Variables:     255
Duplicados de fila: 0
Columnas duplicadas: 0



## 3. Trazabilidad: variable ↔ pregunta ECF

Se construye un catálogo de variables para que cada tabla posterior pueda mostrar:

- referencia técnica;
- pregunta original o resumida;
- bloque de la ECF;
- tipo de variable;
- condición de variable original, pregunta multirrespuesta agrupada o indicador derivado.

Para las preguntas padre multirrespuesta creadas en el master, la descripción se reconstruye a partir de las etiquetas de sus variables hijas originales.


In [23]:

def ecf_block(variable):
    first = str(variable).lower()[:1]

    block_names = {
        "a": "A — Características sociodemográficas",
        "b": "B — Productos, canales y ahorro",
        "c": "C — Fuentes de ingresos",
        "d": "D — Actitudes y comportamiento financiero",
        "e": "E — Competencias financieras",
        "f": "F — Decisiones financieras del hogar",
        "i": "I — Vivienda",
        "j": "J — Gasto, crédito y fragilidad financiera",
    }

    return block_names.get(
        first,
        "Indicadores o variables auxiliares",
    )


def clean_question_label(label):
    if pd.isna(label):
        return ""

    label = re.sub(r"\s+", " ", str(label)).strip()
    return label


def shorten_question(label, max_length=145):
    label = clean_question_label(label)

    if len(label) <= max_length:
        return label

    return label[: max_length - 1].rstrip() + "…"


def parent_children(parent_variable):
    match = re.fullmatch(
        r"([a-z]\d{4})x",
        str(parent_variable).lower(),
    )

    if not match:
        return []

    prefix = match.group(1)

    return [
        variable
        for variable in raw_columns
        if re.fullmatch(
            rf"{re.escape(prefix)}[a-z]",
            str(variable).lower(),
        )
    ]


def infer_parent_question(parent_variable):
    children = parent_children(parent_variable)

    if not children:
        return ""

    labels = [
        clean_question_label(
            raw_variable_labels.get(child, "")
        )
        for child in children
    ]
    labels = [label for label in labels if label]

    if not labels:
        return ""

    # En muchas multirrespuestas la parte inicial es común.
    common_prefix = labels[0]

    for label in labels[1:]:
        while (
            common_prefix
            and not label.startswith(common_prefix)
        ):
            common_prefix = common_prefix[:-1]

    common_prefix = common_prefix.rstrip(
        " :-;,.0123456789"
    )

    if len(common_prefix) >= 25:
        return common_prefix

    # Si no hay prefijo suficientemente claro,
    # se identifica como pregunta agrupada.
    return (
        f"Pregunta multirrespuesta agrupada a partir de: "
        f"{', '.join(children)}"
    )


indicator_descriptions = {}

if {
    "Variable",
    "Definición",
}.issubset(indicator_dictionary.columns):
    indicator_descriptions = dict(
        zip(
            indicator_dictionary["Variable"].astype(str),
            indicator_dictionary["Definición"].astype(str),
        )
    )


def variable_metadata(variable):
    variable = str(variable)

    if variable in raw_variable_labels:
        variable_type = "Original"
        full_question = clean_question_label(
            raw_variable_labels.get(variable, "")
        )
        source = variable

    elif re.fullmatch(r"[a-z]\d{4}x", variable.lower()):
        variable_type = "Multirrespuesta agrupada"
        full_question = infer_parent_question(variable)
        source = ", ".join(parent_children(variable))

    else:
        variable_type = "Indicador derivado"
        full_question = clean_question_label(
            indicator_descriptions.get(variable, "")
        )
        source = "Variables fuente documentadas en el master"

    return {
        "variable": variable,
        "bloque": ecf_block(variable),
        "tipo_variable": variable_type,
        "pregunta_ecf_resumida": shorten_question(
            full_question
        ),
        "pregunta_ecf_completa": full_question,
        "variables_fuente": source,
        "dtype": str(df[variable].dtype),
        "n_no_nulos": int(df[variable].notna().sum()),
        "pct_nulos": round(
            df[variable].isna().mean() * 100,
            2,
        ),
        "n_unicos": int(df[variable].nunique(dropna=True)),
    }


variable_catalog = pd.DataFrame(
    variable_metadata(variable)
    for variable in df.columns
)

display(variable_catalog.head(20))


,variable,bloque,tipo_variable,pregunta_ecf_resumida,pregunta_ecf_completa,variables_fuente,dtype,n_no_nulos,pct_nulos,n_unicos
0,ccaaf,C — Fuentes de ingresos,Original,ccaaf: codigos de comunidad autonoma de residencia,ccaaf: codigos de comunidad autonoma de residencia,ccaaf,str,7764,0.0,17
1,a0000,A — Características sociodemográficas,Original,"a0000: es requisito que pregunte su genero, es vd. hombre o mujer?","a0000: es requisito que pregunte su genero, es vd. hombre o mujer?",a0000,str,7764,0.0,2
2,a04,A — Características sociodemográficas,Original,a04: edad calculada,a04: edad calculada,a04,str,7764,0.0,64
3,a0100,A — Características sociodemográficas,Original,a0100: en que pais nacio?,a0100: en que pais nacio?,a0100,str,7764,0.0,2
4,a0320,A — Características sociodemográficas,Original,a0320: nacio alguno de sus padres fuera de espania?,a0320: nacio alguno de sus padres fuera de espania?,a0320,str,7764,0.0,3
5,a0910,A — Características sociodemográficas,Original,a0910: con cuantas personas vive habitualmente (incluyendose a si mismo)?,a0910: con cuantas personas vive habitualmente (incluyendose a si mismo)?,a0910,int64,7764,0.0,10
6,a1030,A — Características sociodemográficas,Original,a1030: cual es su estado civil actual?,a1030: cual es su estado civil actual?,a1030,str,7764,0.0,7
7,a1100,A — Características sociodemográficas,Original,a1100: cual es el mayor nivel de formacion que ha alcanzado?,a1100: cual es el mayor nivel de formacion que ha alcanzado?,a1100,str,7764,0.0,9
8,a1400,A — Características sociodemográficas,Original,"a1400: cuantos libros habia, donde vivia cuando tenia 10 anios?","a1400: cuantos libros habia, donde vivia cuando tenia 10 anios?",a1400,str,7764,0.0,7
9,a1410,A — Características sociodemográficas,Original,a1410: le explicaron sus padres como encargarse de asuntos financieros?,a1410: le explicaron sus padres como encargarse de asuntos financieros?,a1410,str,7764,0.0,3


### Buscador de variables por concepto

In [24]:

def search_variables(*terms, max_results=50):
    """Busca términos en nombre, pregunta, descripción y bloque."""

    if not terms:
        return variable_catalog.copy()

    mask = pd.Series(
        True,
        index=variable_catalog.index,
    )

    searchable = (
        variable_catalog[
            [
                "variable",
                "bloque",
                "pregunta_ecf_resumida",
                "pregunta_ecf_completa",
            ]
        ]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.lower()
    )

    for term in terms:
        mask &= searchable.str.contains(
            str(term).lower(),
            regex=False,
        )

    return variable_catalog.loc[mask].head(max_results)


# Ejemplos:
display(search_variables("ahorro"))
display(search_variables("crédito"))
display(search_variables("banca"))


,variable,bloque,tipo_variable,pregunta_ecf_resumida,pregunta_ecf_completa,variables_fuente,dtype,n_no_nulos,pct_nulos,n_unicos
18,b0100,"B — Productos, canales y ahorro",Original,"b0100: poseen cuentas corrientes, libretas u otros depositos...?","b0100: poseen cuentas corrientes, libretas u otros depositos...?",b0100,str,7764,0.00,5
19,b0208,"B — Productos, canales y ahorro",Original,b0208: ha oido hablar de las cuentas o depositos de ahorro o a plazo...?,b0208: ha oido hablar de las cuentas o depositos de ahorro o a plazo...?,b0208,str,7764,0.00,3
20,b0308,"B — Productos, canales y ahorro",Original,"b0308: en la actualidad tiene, personal o conjuntamente, alguna cuenta...?","b0308: en la actualidad tiene, personal o conjuntamente, alguna cuenta...?",b0308,str,7764,0.00,5
21,b0408,"B — Productos, canales y ahorro",Original,"b0408: en los ultimos 2 anios ha contratado,(...) cuenta o deposito de ahorro...","b0408: en los ultimos 2 anios ha contratado,(...) cuenta o deposito de ahorro...",b0408,str,7764,0.00,5
22,b0201,"B — Productos, canales y ahorro",Original,b0201: ha oido hablar de las hipotecas?,b0201: ha oido hablar de las hipotecas?,b0201,str,7764,0.00,2
23,b0301,"B — Productos, canales y ahorro",Original,"b0301: en la actualidad tiene, personal o conjuntamente, alguna hipoteca?","b0301: en la actualidad tiene, personal o conjuntamente, alguna hipoteca?",b0301,str,7764,0.00,6
24,b0401,"B — Productos, canales y ahorro",Original,"b0401: en los ultimos 2 anios ha contratado,(...) alguna hipoteca...?","b0401: en los ultimos 2 anios ha contratado,(...) alguna hipoteca...?",b0401,str,7764,0.00,6
25,b0202,"B — Productos, canales y ahorro",Original,b0202: ha oido hablar de los planes de pensiones individuales o de empresa?,b0202: ha oido hablar de los planes de pensiones individuales o de empresa?,b0202,str,7764,0.00,3
26,b0302,"B — Productos, canales y ahorro",Original,b0302: en la actualidad tiene algun plan de pensiones individual o de empresa?,b0302: en la actualidad tiene algun plan de pensiones individual o de empresa?,b0302,str,7764,0.00,5
27,b0402,"B — Productos, canales y ahorro",Original,"b0402: en los ultimos 2 anios ha contratado,(...) plan de pensiones...?","b0402: en los ultimos 2 anios ha contratado,(...) plan de pensiones...?",b0402,str,7764,0.00,6


,variable,bloque,tipo_variable,pregunta_ecf_resumida,pregunta_ecf_completa,variables_fuente,dtype,n_no_nulos,pct_nulos,n_unicos
123,j0100,"J — Gasto, crédito y fragilidad financiera",Original,j0100: hace su hogar una planificacion de sus gastos?,j0100: hace su hogar una planificacion de sus gastos?,j0100,str,7764,0.0,5
124,j0110,"J — Gasto, crédito y fragilidad financiera",Original,j0110: usan alguna app bancaria o herramienta de administracion…,j0110: usan alguna app bancaria o herramienta de administracion…,j0110,str,7764,0.0,5
125,j0200,"J — Gasto, crédito y fragilidad financiera",Original,"j0200: en los ultimos 12 meses, han sido sus gastos mayores que sus ingresos?","j0200: en los ultimos 12 meses, han sido sus gastos mayores que sus ingresos?",j0200,str,7764,0.0,5
126,j0400,"J — Gasto, crédito y fragilidad financiera",Original,j0400: durante cuanto tiempo podria(n) hacer frente a sus gastos corrientes...?,j0400: durante cuanto tiempo podria(n) hacer frente a sus gastos corrientes...?,j0400,str,7764,0.0,10
127,j0500,"J — Gasto, crédito y fragilidad financiera",Original,"j0500: cuanto gastan en promedio en comida, dentro y fuera de casa?","j0500: cuanto gastan en promedio en comida, dentro y fuera de casa?",j0500,str,7764,0.0,110
128,j0600,"J — Gasto, crédito y fragilidad financiera",Original,"j0600: este gasto en comida del que me acaba de hablar, hace referencia a:","j0600: este gasto en comida del que me acaba de hablar, hace referencia a:",j0600,str,7764,0.0,3
129,j0700,"J — Gasto, crédito y fragilidad financiera",Original,j0700: cuanto gasto su hogar en educacion durante el ultimo anio?,j0700: cuanto gasto su hogar en educacion durante el ultimo anio?,j0700,str,7764,0.0,213
130,j0810,"J — Gasto, crédito y fragilidad financiera",Original,j0810: alguna de estas personas no forman parte de su hogar?,j0810: alguna de estas personas no forman parte de su hogar?,j0810,str,7764,0.0,4
131,j0920,"J — Gasto, crédito y fragilidad financiera",Original,j0920: considero pedir algun tipo de estos prestamos en los ultimos dos anios?,j0920: considero pedir algun tipo de estos prestamos en los ultimos dos anios?,j0920,str,7764,0.0,7
132,j1000,"J — Gasto, crédito y fragilidad financiera",Original,"j1000: en los ultimos 12 meses, han retrasado el pago de alguna de sus deudas...","j1000: en los ultimos 12 meses, han retrasado el pago de alguna de sus deudas...",j1000,str,7764,0.0,5


,variable,bloque,tipo_variable,pregunta_ecf_resumida,pregunta_ecf_completa,variables_fuente,dtype,n_no_nulos,pct_nulos,n_unicos
124,j0110,"J — Gasto, crédito y fragilidad financiera",Original,j0110: usan alguna app bancaria o herramienta de administracion…,j0110: usan alguna app bancaria o herramienta de administracion…,j0110,str,7764,0.0,5
133,j1100,"J — Gasto, crédito y fragilidad financiera",Original,"j1100: estas deudas, son deudas contraidas con una entidad financiera o bancaria","j1100: estas deudas, son deudas contraidas con una entidad financiera o bancaria",j1100,str,7764,0.0,4
221,usa_banca_digital,Indicadores o variables auxiliares,Indicador derivado,Uso de ordenador/tablet o app móvil bancaria.,Uso de ordenador/tablet o app móvil bancaria.,Variables fuente documentadas en el master,int64,7764,0.0,2
222,pref_banca_digital,Indicadores o variables auxiliares,Indicador derivado,Preferencia por ordenador/tablet o app móvil bancaria.,Preferencia por ordenador/tablet o app móvil bancaria.,Variables fuente documentadas en el master,int64,7764,0.0,2
223,usa_pago_digital,Indicadores o variables auxiliares,Indicador derivado,Uso de aplicaciones o banca online para realizar pagos.,Uso de aplicaciones o banca online para realizar pagos.,Variables fuente documentadas en el master,int64,7764,0.0,2



## 4. Marco analítico: preguntas antes que gráficos

La investigación se organiza alrededor de seis líneas. Cada línea debe terminar en:

1. un hallazgo verificable;
2. los segmentos afectados;
3. una implicación potencial para Banco Atlas;
4. una visualización posterior que comunique el resultado.

### Preguntas principales

| Línea | Pregunta ECF | Posible conexión con Banco Atlas |
|---|---|---|
| Canales | ¿Qué perfiles utilizan o prefieren la banca digital? | Adopción digital, onboarding, acompañamiento y estrategia omnicanal |
| Productos | ¿Qué segmentos presentan menor tenencia o diversidad de productos? | Venta cruzada, productos de entrada y paquetes adaptados |
| Ahorro | ¿Quién ahorra, por qué vías y con qué grado de formalización? | Cuentas de ahorro, automatización, inversión y educación financiera |
| Planificación | ¿Qué perfiles muestran más disciplina y planificación? | Herramientas de presupuesto, alertas y programas de acompañamiento |
| Fragilidad | ¿Qué grupos tienen más dificultad para afrontar gastos o acceder al crédito? | Prevención, refinanciación responsable y productos de liquidez |
| Vivienda y crédito | ¿Qué barreras de acceso aparecen y en qué segmentos? | Hipoteca, ahorro para entrada, simuladores y asesoramiento |


In [25]:

research_questions = pd.DataFrame([
    {
        "id": "Q1",
        "tema": "Canales",
        "pregunta": (
            "¿Qué segmentos usan banca digital y cuáles mantienen "
            "preferencia por canales presenciales?"
        ),
        "variables_resultado_candidatas": (
            "usa_banca_digital, pref_banca_digital, "
            "n_canales_uso_banco"
        ),
        "segmentaciones": (
            "edad, educación, ingresos, situación laboral, CCAA"
        ),
        "implicación_atlas": (
            "Diseño omnicanal, apoyo a la adopción y comunicaciones "
            "diferenciadas."
        ),
    },
    {
        "id": "Q2",
        "tema": "Tenencia de productos",
        "pregunta": (
            "¿Qué perfiles tienen menor diversidad de productos "
            "financieros y qué productos faltan?"
        ),
        "variables_resultado_candidatas": (
            "n_productos_financieros, tiene_vehiculo_ahorro, "
            "tiene_exposicion_credito, tiene_seguro"
        ),
        "segmentaciones": (
            "edad, ingresos, educación, empleo, hogar"
        ),
        "implicación_atlas": (
            "Productos de entrada, venta cruzada responsable y "
            "paquetes según etapa vital."
        ),
    },
    {
        "id": "Q3",
        "tema": "Ahorro",
        "pregunta": (
            "¿Qué segmentos ahorran menos y cuáles dependen de "
            "mecanismos informales?"
        ),
        "variables_resultado_candidatas": (
            "ahorra_12m, n_vehiculos_ahorro, ahorro_formal, "
            "ahorro_informal"
        ),
        "segmentaciones": (
            "ingresos, edad, empleo, educación, composición del hogar"
        ),
        "implicación_atlas": (
            "Ahorro automático, productos sencillos y transición "
            "desde ahorro informal."
        ),
    },
    {
        "id": "Q4",
        "tema": "Planificación",
        "pregunta": (
            "¿Cómo se relacionan disciplina, planificación y "
            "preocupación financiera con el ahorro?"
        ),
        "variables_resultado_candidatas": (
            "score_disciplina_financiera, "
            "score_planificacion_financiera, "
            "score_preocupacion_financiera"
        ),
        "segmentaciones": (
            "ahorro, ingresos, edad, productos, fragilidad"
        ),
        "implicación_atlas": (
            "Herramientas de presupuesto, nudges y alertas "
            "personalizadas."
        ),
    },
    {
        "id": "Q5",
        "tema": "Fragilidad financiera",
        "pregunta": (
            "¿Qué perfiles presentan mayor fragilidad y qué "
            "mecanismos usan para financiar imprevistos?"
        ),
        "variables_resultado_candidatas": (
            "score_fragilidad_financiera, "
            "financiacion_ahorros_activos, "
            "financiacion_credito_formal, "
            "financiacion_credito_informal, "
            "financiacion_estres_pago"
        ),
        "segmentaciones": (
            "ingresos, empleo, edad, hogar, ahorro, acceso al crédito"
        ),
        "implicación_atlas": (
            "Detección preventiva, colchón de emergencia y soluciones "
            "de liquidez responsable."
        ),
    },
    {
        "id": "Q6",
        "tema": "Vivienda y crédito",
        "pregunta": (
            "¿Qué segmentos afrontan mayores barreras para adquirir "
            "vivienda o acceder al crédito?"
        ),
        "variables_resultado_candidatas": (
            "barrera_acceso_vivienda, "
            "n_dificultades_compra_vivienda, "
            "restriccion_acceso_credito"
        ),
        "segmentaciones": (
            "edad, ingresos, empleo, hogar, CCAA"
        ),
        "implicación_atlas": (
            "Ahorro para entrada, simulación hipotecaria, "
            "asesoramiento y evaluación responsable."
        ),
    },
])

display(research_questions)


,id,tema,pregunta,variables_resultado_candidatas,segmentaciones,implicación_atlas
0,Q1,Canales,¿Qué segmentos usan banca digital y cuáles mantienen preferencia por canales presenciales?,"usa_banca_digital, pref_banca_digital, n_canales_uso_banco","edad, educación, ingresos, situación laboral, CCAA","Diseño omnicanal, apoyo a la adopción y comunicaciones diferenciadas."
1,Q2,Tenencia de productos,¿Qué perfiles tienen menor diversidad de productos financieros y qué productos faltan?,"n_productos_financieros, tiene_vehiculo_ahorro, tiene_exposicion_credito, tiene_seguro","edad, ingresos, educación, empleo, hogar","Productos de entrada, venta cruzada responsable y paquetes según etapa vital."
2,Q3,Ahorro,¿Qué segmentos ahorran menos y cuáles dependen de mecanismos informales?,"ahorra_12m, n_vehiculos_ahorro, ahorro_formal, ahorro_informal","ingresos, edad, empleo, educación, composición del hogar","Ahorro automático, productos sencillos y transición desde ahorro informal."
3,Q4,Planificación,"¿Cómo se relacionan disciplina, planificación y preocupación financiera con el ahorro?","score_disciplina_financiera, score_planificacion_financiera, score_preocupacion_financiera","ahorro, ingresos, edad, productos, fragilidad","Herramientas de presupuesto, nudges y alertas personalizadas."
4,Q5,Fragilidad financiera,¿Qué perfiles presentan mayor fragilidad y qué mecanismos usan para financiar imprevistos?,"score_fragilidad_financiera, financiacion_ahorros_activos, financiacion_credito_formal, financiacion_credito_informal, financiacion_estres_pago","ingresos, empleo, edad, hogar, ahorro, acceso al crédito","Detección preventiva, colchón de emergencia y soluciones de liquidez responsable."
5,Q6,Vivienda y crédito,¿Qué segmentos afrontan mayores barreras para adquirir vivienda o acceder al crédito?,"barrera_acceso_vivienda, n_dificultades_compra_vivienda, restriccion_acceso_credito","edad, ingresos, empleo, hogar, CCAA","Ahorro para entrada, simulación hipotecaria, asesoramiento y evaluación responsable."


## 5. Identificación automática de variables de comportamiento

In [26]:

THEME_PATTERNS = {
    "Canales y digitalización": [
        "canal",
        "banca digital",
        "internet",
        "móvil",
        "oficina",
        "cajero",
        "pago digital",
    ],
    "Productos financieros": [
        "producto",
        "cuenta",
        "tarjeta",
        "seguro",
        "pensión",
        "fondo",
        "acciones",
        "bonos",
        "hipoteca",
        "préstamo",
    ],
    "Ahorro": [
        "ahorra",
        "ahorro",
        "vehículo",
    ],
    "Ingresos": [
        "ingreso",
        "renta",
        "fuente",
    ],
    "Actitudes y planificación": [
        "disciplina",
        "planificación",
        "preocupación",
        "destino",
        "presupuesto",
        "gasto",
    ],
    "Fragilidad y financiación": [
        "fragilidad",
        "financiación",
        "financiacion",
        "dificultad",
        "imprevisto",
        "crédito informal",
        "restricción",
        "restriccion",
    ],
    "Vivienda": [
        "vivienda",
        "precio",
        "alquiler",
        "compra",
    ],
}


def classify_theme(row):
    searchable = " ".join([
        str(row["variable"]),
        str(row["pregunta_ecf_completa"]),
        str(row["bloque"]),
    ]).lower()

    matches = []

    for theme, patterns in THEME_PATTERNS.items():
        if any(pattern in searchable for pattern in patterns):
            matches.append(theme)

    return " | ".join(matches) if matches else "Otros"


variable_catalog["tema_analítico"] = variable_catalog.apply(
    classify_theme,
    axis=1,
)

theme_summary = (
    variable_catalog["tema_analítico"]
    .value_counts()
    .rename_axis("tema")
    .reset_index(name="n_variables")
)

display(theme_summary)


,tema,n_variables
0,Otros,94
1,Canales y digitalización | Productos financieros | Ahorro,51
2,Productos financieros,20
3,Actitudes y planificación | Fragilidad y financiación,19
4,Vivienda,15
5,Ingresos,14
6,Ahorro,5
7,Productos financieros | Ahorro,5
8,Fragilidad y financiación,5
9,Productos financieros | Vivienda,4


## 6. Auditoría inicial del dataset

In [27]:

audit = variable_catalog[
    [
        "variable",
        "bloque",
        "tipo_variable",
        "pregunta_ecf_resumida",
        "dtype",
        "n_no_nulos",
        "pct_nulos",
        "n_unicos",
        "tema_analítico",
    ]
].copy()

audit["posible_identificador"] = (
    (audit["n_unicos"] == len(df))
    & (audit["pct_nulos"] == 0)
)

audit["constante"] = audit["n_unicos"] <= 1
audit["alta_cardinalidad"] = (
    audit["n_unicos"] > max(50, len(df) * 0.20)
)

display(
    audit.sort_values(
        ["pct_nulos", "n_unicos"],
        ascending=[False, True],
    ).head(40)
)


,variable,bloque,tipo_variable,pregunta_ecf_resumida,dtype,n_no_nulos,pct_nulos,n_unicos,tema_analítico,posible_identificador,constante,alta_cardinalidad
119,i0400b,I — Vivienda,Original,i0400b: como evolucionara el valor de la vivienda? b. caida entre 2% y 6%,float64,7494,3.48,11,Vivienda,False,False,False
118,i0400a,I — Vivienda,Original,i0400a: como evolucionara el valor de la vivienda? a. caida mas del 6%,float64,7494,3.48,12,Vivienda,False,False,False
120,i0400c,I — Vivienda,Original,i0400c: como evolucionara el valor de la vivienda? c. estable (<+-2%),float64,7494,3.48,12,Vivienda,False,False,False
121,i0400d,I — Vivienda,Original,i0400d: como evolucionara el valor de la vivienda? d. subida entre 2% y 6%,float64,7494,3.48,12,Vivienda,False,False,False
122,i0400e,I — Vivienda,Original,i0400e: como evolucionara el valor de la vivienda? e. subida mas del 6%,float64,7494,3.48,12,Vivienda,False,False,False
247,expectativa_subida_precio_vivienda,E — Competencias financieras,Indicador derivado,Puntos asignados a escenarios de subida del precio.,float64,7494,3.48,12,Vivienda,False,False,False
85,d0610e,D — Actitudes y comportamiento financiero,Original,d0610e: como utilizaria ese ingreso extra en los proximos 12 meses?: e. otros,float64,7689,0.97,12,Ingresos,False,False,False
81,d0610a,D — Actitudes y comportamiento financiero,Original,d0610a: como utilizaria ese ingreso extra en los proximos 12 meses?: a. producto,float64,7689,0.97,13,Productos financieros | Ingresos,False,False,False
82,d0610b,D — Actitudes y comportamiento financiero,Original,d0610b: como utilizaria ese ingreso extra en los proximos 12 meses?: b. producto,float64,7689,0.97,13,Productos financieros | Ingresos,False,False,False
83,d0610c,D — Actitudes y comportamiento financiero,Original,d0610c: como utilizaria ese ingreso extra en los proximos 12 meses?: c. ahorro,float64,7689,0.97,13,Ahorro | Ingresos,False,False,False


### Comprobaciones críticas

In [28]:

critical_checks = pd.DataFrame([
    {
        "comprobación": "Filas duplicadas",
        "resultado": int(df.duplicated().sum()),
        "estado": (
            "OK"
            if df.duplicated().sum() == 0
            else "REVISAR"
        ),
    },
    {
        "comprobación": "Columnas duplicadas",
        "resultado": int(df.columns.duplicated().sum()),
        "estado": (
            "OK"
            if df.columns.duplicated().sum() == 0
            else "REVISAR"
        ),
    },
    {
        "comprobación": "Columnas completamente vacías",
        "resultado": int(df.isna().all().sum()),
        "estado": (
            "OK"
            if df.isna().all().sum() == 0
            else "REVISAR"
        ),
    },
    {
        "comprobación": "Columnas constantes",
        "resultado": int((df.nunique(dropna=False) <= 1).sum()),
        "estado": (
            "OK"
            if (df.nunique(dropna=False) <= 1).sum() == 0
            else "REVISAR"
        ),
    },
])

display(critical_checks)


,comprobación,resultado,estado
0,Filas duplicadas,0,OK
1,Columnas duplicadas,0,OK
2,Columnas completamente vacías,0,OK
3,Columnas constantes,0,OK



## 7. Variables de segmentación

Esta sección localiza las variables demográficas disponibles. La selección definitiva debe validarse con sus etiquetas ECF y no únicamente con el nombre técnico.


In [29]:

SEGMENTATION_TERMS = [
    "sexo",
    "género",
    "edad",
    "comunidad autónoma",
    "comunidad autonoma",
    "nacimiento",
    "nacionalidad",
    "estado civil",
    "educación",
    "educacion",
    "estudios",
    "situación laboral",
    "situacion laboral",
    "ocupación",
    "ocupacion",
    "ingreso",
    "renta",
    "hogar",
    "miembros",
    "hijos",
]


def contains_any(text, terms):
    text = str(text).lower()
    return any(term in text for term in terms)


segmentation_catalog = variable_catalog.loc[
    variable_catalog.apply(
        lambda row: contains_any(
            " ".join([
                row["variable"],
                row["pregunta_ecf_completa"],
                row["bloque"],
            ]),
            SEGMENTATION_TERMS,
        ),
        axis=1,
    )
].copy()

display(
    segmentation_catalog[
        [
            "variable",
            "pregunta_ecf_resumida",
            "tipo_variable",
            "n_unicos",
            "pct_nulos",
        ]
    ].sort_values("variable")
)


,variable,pregunta_ecf_resumida,tipo_variable,n_unicos,pct_nulos
2,a04,a04: edad calculada,Original,64,0.00
6,a1030,a1030: cual es su estado civil actual?,Original,7,0.00
10,a1500,a1500: cual es su situacion laboral actual?,Original,9,0.00
11,a1510,"a1510. si tiene más de una situacion laboral, cual es la secundaria?",Original,12,0.00
34,b0205,b0205: ha oido hablar de los activos de renta fija publica/privada...?,Original,3,0.00
35,b0305,"b0305: en la actualidad tiene, (...) algun activo de renta fija publica/privada?",Original,6,0.00
36,b0405,"b0405: en los ultimos 2 anios ha adquirido,(...) algun activo de renta fija...?",Original,5,0.00
57,c0100,c0100: como cree que esta planificando su jubilacion o vejez?,Original,10,0.00
209,c0200x,"Pregunta multirrespuesta agrupada a partir de: c0200a, c0200b, c0200c, c0200d, c0200e, c0200f, c0200g, c0200h, c0200i, c0200j, c0200k",Multirrespuesta agrupada,382,0.00
58,c0300,c0300: a que edad espera jubilarse?,Original,43,0.00



## 8. Funciones de tablas con trazabilidad

Las tablas siguientes incorporan siempre la referencia de variable y la pregunta ECF resumida.


In [30]:

def question_for(variable):
    match = variable_catalog.loc[
        variable_catalog["variable"] == variable,
        "pregunta_ecf_resumida",
    ]

    return (
        match.iloc[0]
        if not match.empty
        else ""
    )


def frequency_table(variable, include_missing=True):
    if variable not in df.columns:
        raise KeyError(
            f"La variable '{variable}' no existe en el master."
        )

    series = df[variable]

    counts = series.value_counts(
        dropna=not include_missing
    )

    result = counts.rename("n").reset_index()
    result.columns = ["respuesta", "n"]
    result["porcentaje"] = (
        result["n"] / len(df) * 100
    ).round(2)

    result.insert(0, "variable", variable)
    result.insert(
        1,
        "pregunta_ecf",
        question_for(variable),
    )

    return result


def numeric_summary(variable):
    if variable not in df.columns:
        raise KeyError(
            f"La variable '{variable}' no existe en el master."
        )

    series = pd.to_numeric(
        df[variable],
        errors="coerce",
    )

    result = pd.DataFrame({
        "variable": [variable],
        "pregunta_ecf": [question_for(variable)],
        "n": [int(series.notna().sum())],
        "media": [series.mean()],
        "desv_std": [series.std()],
        "mínimo": [series.min()],
        "p25": [series.quantile(0.25)],
        "mediana": [series.median()],
        "p75": [series.quantile(0.75)],
        "máximo": [series.max()],
    })

    return result.round(3)


def segment_rate(
    outcome,
    segment,
    positive_value=1,
    min_group_n=30,
):
    required = {outcome, segment}

    missing = required - set(df.columns)

    if missing:
        raise KeyError(
            f"Faltan variables: {sorted(missing)}"
        )

    work = df[[outcome, segment]].copy()
    work = work.dropna()

    work["_positive"] = (
        work[outcome] == positive_value
    ).astype(int)

    result = (
        work.groupby(segment, dropna=False)
        .agg(
            n=("_positive", "size"),
            tasa=("_positive", "mean"),
        )
        .reset_index()
    )

    result = result.loc[result["n"] >= min_group_n]
    result["tasa_pct"] = (
        result["tasa"] * 100
    ).round(2)
    result = result.drop(columns="tasa")

    result.insert(0, "resultado", outcome)
    result.insert(
        1,
        "pregunta_resultado",
        question_for(outcome),
    )
    result.insert(2, "segmentación", segment)
    result.insert(
        3,
        "pregunta_segmentación",
        question_for(segment),
    )

    return result.sort_values(
        "tasa_pct",
        ascending=False,
    )


## 9. Diagnóstico inicial de indicadores clave

In [31]:

CORE_INDICATORS = [
    "n_canales_uso_banco",
    "usa_banca_digital",
    "pref_banca_digital",
    "usa_pago_digital",
    "n_productos_financieros",
    "tiene_vehiculo_ahorro",
    "tiene_exposicion_credito",
    "tiene_seguro",
    "ahorra_12m",
    "n_vehiculos_ahorro",
    "ahorro_formal",
    "ahorro_informal",
    "n_fuentes_ingreso",
    "ingreso_por_activos",
    "dependencia_ingresos_familiares",
    "score_disciplina_financiera",
    "score_preocupacion_financiera",
    "score_planificacion_financiera",
    "puntos_destino_consumo",
    "puntos_destino_ahorro",
    "puntos_destino_deuda",
    "barrera_acceso_vivienda",
    "n_dificultades_compra_vivienda",
    "financiacion_ahorros_activos",
    "financiacion_credito_informal",
    "financiacion_credito_formal",
    "financiacion_estres_pago",
    "restriccion_acceso_credito",
    "score_fragilidad_financiera",
]

available_core_indicators = [
    variable
    for variable in CORE_INDICATORS
    if variable in df.columns
]

missing_core_indicators = sorted(
    set(CORE_INDICATORS)
    - set(available_core_indicators)
)

print(
    f"Indicadores clave disponibles: "
    f"{len(available_core_indicators)}"
)
print(
    "Indicadores clave no encontrados:",
    missing_core_indicators,
)

core_indicator_catalog = variable_catalog.loc[
    variable_catalog["variable"].isin(
        available_core_indicators
    )
].copy()

display(
    core_indicator_catalog[
        [
            "variable",
            "pregunta_ecf_resumida",
            "dtype",
            "n_unicos",
            "pct_nulos",
        ]
    ]
)


Indicadores clave disponibles: 29
Indicadores clave no encontrados: []


,variable,pregunta_ecf_resumida,dtype,n_unicos,pct_nulos
220,n_canales_uso_banco,Número de canales utilizados para relacionarse con el banco.,int64,7,0.00
221,usa_banca_digital,Uso de ordenador/tablet o app móvil bancaria.,int64,2,0.00
222,pref_banca_digital,Preferencia por ordenador/tablet o app móvil bancaria.,int64,2,0.00
223,usa_pago_digital,Uso de aplicaciones o banca online para realizar pagos.,int64,2,0.00
224,n_productos_financieros,Número de familias/productos financieros actualmente contratados.,int64,10,0.00
225,tiene_vehiculo_ahorro,Tenencia de algún producto de ahorro o inversión.,int64,2,0.00
226,tiene_exposicion_credito,"Tenencia de hipoteca, préstamo personal o tarjeta de crédito.",int64,2,0.00
227,tiene_seguro,Tenencia de seguro de vida o seguro médico.,int64,2,0.00
228,ahorra_12m,Ha utilizado al menos un mecanismo de ahorro en los últimos 12 meses.,int64,2,0.00
229,n_vehiculos_ahorro,Número de mecanismos de ahorro utilizados.,int64,7,0.00


In [32]:

numeric_core = [
    variable
    for variable in available_core_indicators
    if pd.api.types.is_numeric_dtype(df[variable])
    and df[variable].nunique(dropna=True) > 2
]

binary_or_categorical_core = [
    variable
    for variable in available_core_indicators
    if variable not in numeric_core
]

numeric_diagnostics = pd.concat(
    [
        numeric_summary(variable)
        for variable in numeric_core
    ],
    ignore_index=True,
) if numeric_core else pd.DataFrame()

display(numeric_diagnostics)

for variable in binary_or_categorical_core:
    display(frequency_table(variable))


,variable,pregunta_ecf,n,media,desv_std,mínimo,p25,mediana,p75,máximo
0,n_canales_uso_banco,Número de canales utilizados para relacionarse con el banco.,7764,2.990,1.397,0.0,2.0,3.00,4.00,6.0
1,n_productos_financieros,Número de familias/productos financieros actualmente contratados.,7764,2.494,1.902,0.0,1.0,2.00,4.00,9.0
2,n_vehiculos_ahorro,Número de mecanismos de ahorro utilizados.,7764,1.126,0.946,0.0,0.0,1.00,2.00,6.0
3,n_fuentes_ingreso,Número de fuentes actuales o previstas de ingresos.,7764,2.464,1.403,0.0,1.0,2.00,3.00,11.0
4,score_disciplina_financiera,"Media 1–5 de control, pago puntual, vigilancia e información.",7763,4.254,0.571,1.0,4.0,4.25,4.75,5.0
5,score_preocupacion_financiera,"Media 1–5 de preocupación, endeudamiento e inquietud.",7764,3.292,0.718,1.0,2.8,3.20,3.80,5.0
6,score_planificacion_financiera,Media 1–5 orientada al futuro; ítems negativos invertidos.,7763,3.593,0.729,1.0,3.2,3.60,4.20,5.0
7,puntos_destino_consumo,Puntos del ingreso extra destinados a consumo.,7689,4.071,3.933,-10.0,2.0,5.00,7.00,10.0
8,puntos_destino_ahorro,Puntos del ingreso extra destinados a ahorro.,7689,2.913,3.191,-5.0,0.0,2.00,5.00,10.0
9,puntos_destino_deuda,Puntos del ingreso extra destinados a pagar deuda.,7689,1.402,2.912,-5.0,0.0,0.00,2.00,10.0


,variable,pregunta_ecf,respuesta,n,porcentaje
0,usa_banca_digital,Uso de ordenador/tablet o app móvil bancaria.,1,5818,74.94
1,usa_banca_digital,Uso de ordenador/tablet o app móvil bancaria.,0,1946,25.06


,variable,pregunta_ecf,respuesta,n,porcentaje
0,pref_banca_digital,Preferencia por ordenador/tablet o app móvil bancaria.,1,4242,54.64
1,pref_banca_digital,Preferencia por ordenador/tablet o app móvil bancaria.,0,3522,45.36


,variable,pregunta_ecf,respuesta,n,porcentaje
0,usa_pago_digital,Uso de aplicaciones o banca online para realizar pagos.,1,5484,70.63
1,usa_pago_digital,Uso de aplicaciones o banca online para realizar pagos.,0,2280,29.37


,variable,pregunta_ecf,respuesta,n,porcentaje
0,tiene_vehiculo_ahorro,Tenencia de algún producto de ahorro o inversión.,0,4356,56.11
1,tiene_vehiculo_ahorro,Tenencia de algún producto de ahorro o inversión.,1,3408,43.89


...

,variable,pregunta_ecf,respuesta,n,porcentaje
0,financiacion_ahorros_activos,Cubrió el déficit utilizando ahorros o vendiendo activos.,0,6744,86.86
1,financiacion_ahorros_activos,Cubrió el déficit utilizando ahorros o vendiendo activos.,1,1020,13.14


,variable,pregunta_ecf,respuesta,n,porcentaje
0,financiacion_credito_informal,"Cubrió el déficit mediante familia, adelantos o proveedores.",0,7205,92.8
1,financiacion_credito_informal,"Cubrió el déficit mediante familia, adelantos o proveedores.",1,559,7.2


,variable,pregunta_ecf,respuesta,n,porcentaje
0,financiacion_credito_formal,Cubrió el déficit mediante crédito o financiación formal.,0,7417,95.53
1,financiacion_credito_formal,Cubrió el déficit mediante crédito o financiación formal.,1,347,4.47


,variable,pregunta_ecf,respuesta,n,porcentaje
0,financiacion_estres_pago,Utilizó descubierto no autorizado o retrasó pagos.,0,7545,97.18
1,financiacion_estres_pago,Utilizó descubierto no autorizado o retrasó pagos.,1,219,2.82


,variable,pregunta_ecf,respuesta,n,porcentaje
0,restriccion_acceso_credito,"Rechazo, concesión parcial o autoexclusión crediticia.",0,7259,93.5
1,restriccion_acceso_credito,"Rechazo, concesión parcial o autoexclusión crediticia.",1,505,6.5



## 10. Cribado de relaciones candidatas

Este cribado no constituye todavía una conclusión. Sirve para encontrar relaciones que merezcan una comprobación más profunda.

Se revisarán:

- correlaciones entre indicadores numéricos;
- diferencias de indicadores por segmentos;
- tasas de resultados binarios por segmento;
- tamaños de muestra para evitar destacar grupos demasiado pequeños.


In [33]:

def spearman_screen(
    variables,
    min_complete=100,
    min_abs_correlation=0.10,
):
    existing = [
        variable
        for variable in variables
        if variable in df.columns
    ]

    numeric = df[existing].apply(
        pd.to_numeric,
        errors="coerce",
    )

    rows = []

    for variable_a, variable_b in combinations(
        existing,
        2,
    ):
        pair = numeric[
            [variable_a, variable_b]
        ].dropna()

        if len(pair) < min_complete:
            continue

        correlation = pair.corr(
            method="spearman"
        ).iloc[0, 1]

        if (
            pd.notna(correlation)
            and abs(correlation) >= min_abs_correlation
        ):
            rows.append({
                "variable_1": variable_a,
                "pregunta_1": question_for(variable_a),
                "variable_2": variable_b,
                "pregunta_2": question_for(variable_b),
                "n_completo": len(pair),
                "rho_spearman": round(
                    correlation,
                    3,
                ),
                "intensidad_abs": round(
                    abs(correlation),
                    3,
                ),
            })

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values(
            "intensidad_abs",
            ascending=False,
        )
        .reset_index(drop=True)
    )


behaviour_numeric_candidates = [
    "n_canales_uso_banco",
    "n_productos_financieros",
    "n_vehiculos_ahorro",
    "n_fuentes_ingreso",
    "score_disciplina_financiera",
    "score_preocupacion_financiera",
    "score_planificacion_financiera",
    "puntos_destino_consumo",
    "puntos_destino_ahorro",
    "puntos_destino_deuda",
    "n_dificultades_compra_vivienda",
    "score_fragilidad_financiera",
]

correlation_candidates = spearman_screen(
    behaviour_numeric_candidates,
    min_complete=100,
    min_abs_correlation=0.10,
)

display(correlation_candidates.head(30))


,variable_1,pregunta_1,variable_2,pregunta_2,n_completo,rho_spearman,intensidad_abs
0,n_canales_uso_banco,Número de canales utilizados para relacionarse con el banco.,n_productos_financieros,Número de familias/productos financieros actualmente contratados.,7764,0.394,0.394
1,score_preocupacion_financiera,"Media 1–5 de preocupación, endeudamiento e inquietud.",score_fragilidad_financiera,"Índice 0–4: déficit, impagos, pérdida de empleo y restricción crediticia.",7764,0.365,0.365
2,puntos_destino_consumo,Puntos del ingreso extra destinados a consumo.,puntos_destino_ahorro,Puntos del ingreso extra destinados a ahorro.,7689,-0.334,0.334
3,score_disciplina_financiera,"Media 1–5 de control, pago puntual, vigilancia e información.",score_planificacion_financiera,Media 1–5 orientada al futuro; ítems negativos invertidos.,7762,0.316,0.316
4,n_vehiculos_ahorro,Número de mecanismos de ahorro utilizados.,n_fuentes_ingreso,Número de fuentes actuales o previstas de ingresos.,7764,0.315,0.315
5,n_canales_uso_banco,Número de canales utilizados para relacionarse con el banco.,n_vehiculos_ahorro,Número de mecanismos de ahorro utilizados.,7764,0.301,0.301
6,n_productos_financieros,Número de familias/productos financieros actualmente contratados.,n_vehiculos_ahorro,Número de mecanismos de ahorro utilizados.,7764,0.274,0.274
7,puntos_destino_consumo,Puntos del ingreso extra destinados a consumo.,puntos_destino_deuda,Puntos del ingreso extra destinados a pagar deuda.,7689,-0.257,0.257
8,n_vehiculos_ahorro,Número de mecanismos de ahorro utilizados.,score_preocupacion_financiera,"Media 1–5 de preocupación, endeudamiento e inquietud.",7764,-0.235,0.235
9,score_preocupacion_financiera,"Media 1–5 de preocupación, endeudamiento e inquietud.",puntos_destino_deuda,Puntos del ingreso extra destinados a pagar deuda.,7689,0.221,0.221


## 11. Priorización de preguntas para la Fase 2

In [34]:

priority_matrix = research_questions.copy()

priority_matrix["disponibilidad_variables"] = (
    priority_matrix[
        "variables_resultado_candidatas"
    ]
    .str.split(", ")
    .apply(
        lambda variables: (
            f"{sum(v in df.columns for v in variables)}"
            f"/{len(variables)}"
        )
    )
)

priority_matrix["prioridad_inicial"] = [
    "Alta",
    "Alta",
    "Muy alta",
    "Muy alta",
    "Muy alta",
    "Alta",
]

priority_matrix["razón_prioridad"] = [
    (
        "Permite contrastar la estrategia digital y detectar "
        "necesidades de acompañamiento."
    ),
    (
        "Conecta directamente con propuesta comercial y "
        "diversificación de productos."
    ),
    (
        "El ahorro es un comportamiento central y accionable "
        "para una entidad bancaria."
    ),
    (
        "Puede explicar diferencias de ahorro, productos y "
        "fragilidad."
    ),
    (
        "Tiene conexión directa con riesgo, prevención y "
        "bienestar financiero."
    ),
    (
        "Puede traducirse en productos de ahorro, hipoteca y "
        "asesoramiento."
    ),
]

display(
    priority_matrix[
        [
            "id",
            "tema",
            "pregunta",
            "disponibilidad_variables",
            "prioridad_inicial",
            "razón_prioridad",
        ]
    ]
)


,id,tema,pregunta,disponibilidad_variables,prioridad_inicial,razón_prioridad
0,Q1,Canales,¿Qué segmentos usan banca digital y cuáles mantienen preferencia por canales presenciales?,3/3,Alta,Permite contrastar la estrategia digital y detectar necesidades de acompañamiento.
1,Q2,Tenencia de productos,¿Qué perfiles tienen menor diversidad de productos financieros y qué productos faltan?,4/4,Alta,Conecta directamente con propuesta comercial y diversificación de productos.
2,Q3,Ahorro,¿Qué segmentos ahorran menos y cuáles dependen de mecanismos informales?,4/4,Muy alta,El ahorro es un comportamiento central y accionable para una entidad bancaria.
3,Q4,Planificación,"¿Cómo se relacionan disciplina, planificación y preocupación financiera con el ahorro?",3/3,Muy alta,"Puede explicar diferencias de ahorro, productos y fragilidad."
4,Q5,Fragilidad financiera,¿Qué perfiles presentan mayor fragilidad y qué mecanismos usan para financiar imprevistos?,5/5,Muy alta,"Tiene conexión directa con riesgo, prevención y bienestar financiero."
5,Q6,Vivienda y crédito,¿Qué segmentos afrontan mayores barreras para adquirir vivienda o acceder al crédito?,3/3,Alta,"Puede traducirse en productos de ahorro, hipoteca y asesoramiento."



## 12. Registro de hallazgos

Esta tabla se completará solo cuando una relación haya sido comprobada.

Un hallazgo válido debe incluir:

- evidencia cuantitativa;
- variable y pregunta ECF;
- segmento;
- tamaño de muestra;
- interpretación prudente;
- conexión con Banco Atlas;
- limitación;
- propuesta de visualización.

No se utilizarán expresiones causales como “provoca” o “genera” salvo que el diseño del análisis permita sostenerlas.


In [35]:

insight_register = pd.DataFrame(
    columns=[
        "id_hallazgo",
        "pregunta_análisis",
        "variable_resultado",
        "pregunta_ecf_resultado",
        "segmento_o_variable_relacionada",
        "pregunta_ecf_segmento",
        "evidencia_cuantitativa",
        "n_analizado",
        "interpretación",
        "implicación_para_banco_atlas",
        "limitación",
        "visualización_propuesta",
        "estado",
    ]
)

display(insight_register)


,id_hallazgo,pregunta_análisis,variable_resultado,pregunta_ecf_resultado,segmento_o_variable_relacionada,pregunta_ecf_segmento,evidencia_cuantitativa,n_analizado,interpretación,implicación_para_banco_atlas,limitación,visualización_propuesta,estado



## 13. Próximo paso recomendado

La **Fase 2** debe comenzar por tres historias prioritarias:

### Historia A — Capacidad de ahorro y fragilidad

- ¿Quién ahorra?
- ¿Quién no puede ahorrar?
- ¿Cómo se relaciona el ahorro con la fragilidad?
- ¿Qué mecanismo se utiliza ante un imprevisto?
- ¿Qué solución podría ofrecer Banco Atlas?

### Historia B — Digitalización y brecha de canal

- ¿Quién usa banca digital?
- ¿Quién la usa pero no la prefiere?
- ¿Quién permanece fuera de los canales digitales?
- ¿Qué segmentos requieren acompañamiento o una experiencia omnicanal?

### Historia C — Planificación, productos y comportamiento

- ¿La disciplina y planificación se asocian con mayor ahorro?
- ¿Se relacionan con una cartera más diversificada?
- ¿Qué perfiles combinan baja planificación y elevada fragilidad?
- ¿Qué nudges o herramientas de gestión podrían resultar útiles?

Solo después de validar estas relaciones se diseñarán las visualizaciones.


## 14. Guardado opcional de catálogos de trabajo

In [36]:

# Se dejan preparados, pero no se exportan automáticamente para evitar
# generar archivos innecesarios durante cada ejecución.
#
# Descomenta cuando quieras conservarlos:

# variable_catalog.to_csv(
#     DATA_DIR / "2026-07-20_ECF_2021_02_Variable_Catalog.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# priority_matrix.to_csv(
#     DATA_DIR / "2026-07-20_ECF_2021_02_Research_Questions.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

# insight_register.to_csv(
#     DATA_DIR / "2026-07-20_ECF_2021_02_Insight_Register.csv",
#     index=False,
#     encoding="utf-8-sig",
# )

print("Fase 1 preparada.")


Fase 1 preparada.
